Here I go, embarking on the exciting journey of implementing BERT from scratch, following the guidelines of the original paper! My aim is to truly grasp the intricacies of its architecture and concepts, building practical experience step by step.

### Understanding BERT's Architecture

Before I start coding, I need to remind myself of the core architecture of BERT as described in the paper. BERT, which stands for Bidirectional Encoder Representations from Transformers, is essentially a multi-layer bidirectional Transformer encoder.

Key components I'll be implementing include:

1.  **Input Embeddings**: This involves token embeddings, segment embeddings (to distinguish between two sentences in a pair), and position embeddings (to indicate the position of words in a sequence).
2.  **Transformer Encoder**: BERT uses multiple layers of identical Transformer encoders. Each encoder layer consists of:
    *   A Multi-Head Self-Attention mechanism: This allows the model to weigh the importance of different words in the input sequence when encoding a particular word.
    *   A Feed-Forward Network: A simple fully connected feed-forward network applied to each position independently.
    *   Layer Normalization and Residual Connections: These are applied around both the self-attention and feed-forward sub-layers to aid training stability and convergence.
3.  **Pre-training Tasks**: While I might not fully implement the pre-training from scratch, understanding them is crucial. The paper highlights two main tasks:
    *   **Masked Language Model (MLM)**: Randomly masking some tokens from the input and predicting them based on their context.
    *   **Next Sentence Prediction (NSP)**: Predicting whether two segments are consecutive in the original text.

My implementation will focus on building these components from the ground up, starting with the basic building blocks.

In [1]:
# First, I'm going to import the necessary libraries. I'll primarily use PyTorch for building the model.
import torch
import torch.nn as nn
import math

# I'll also define some hyperparameters for my BERT model, largely based on the 'BERT-Base' configuration described in the paper.
# This will help me keep my implementation consistent with the original design.

class BertConfig:
    # Initialize with default BERT-Base parameters
    def __init__(self,
                 vocab_size=30522,       # Size of the vocabulary (from WordPiece tokenization)
                 hidden_size=768,        # Dimensionality of the encoder layers and the pooler layer
                 num_hidden_layers=12,   # Number of hidden layers in the Transformer encoder
                 num_attention_heads=12, # Number of attention heads for each attention layer
                 intermediate_size=3072, # Dimensionality of the "intermediate" (i.e., feed-forward) layer in the Transformer encoder
                 hidden_act="gelu",      # The activation function in the encoder and pooler
                 hidden_dropout_prob=0.1,# The dropout probability for all fully connected layers in the embeddings, encoder, and pooler
                 attention_probs_dropout_prob=0.1, # The dropout ratio for the attention probabilities
                 max_position_embeddings=512, # The maximum sequence length that this model might ever be used with
                 type_vocab_size=2,      # The vocabulary size of the `token_type_ids` (segment embeddings)
                 initializer_range=0.02  # The standard deviation of the truncated_normal_initializer for initializing all weight matrices
                ):
        self.vocab_size = vocab_size
        self.hidden_size = hidden_size
        self.num_hidden_layers = num_hidden_layers
        self.num_attention_heads = num_attention_heads
        self.intermediate_size = intermediate_size
        self.hidden_act = hidden_act
        self.hidden_dropout_prob = hidden_dropout_prob
        self.attention_probs_dropout_prob = attention_probs_dropout_prob
        self.max_position_embeddings = max_position_embeddings
        self.type_vocab_size = type_vocab_size
        self.initializer_range = initializer_range

# Now I'm creating an instance of my configuration for BERT-Base.
bert_config = BertConfig()

print("BERT-Base configuration initialized:")
for key, value in vars(bert_config).items():
    print(f"  {key}: {value}")


BERT-Base configuration initialized:
  vocab_size: 30522
  hidden_size: 768
  num_hidden_layers: 12
  num_attention_heads: 12
  intermediate_size: 3072
  hidden_act: gelu
  hidden_dropout_prob: 0.1
  attention_probs_dropout_prob: 0.1
  max_position_embeddings: 512
  type_vocab_size: 2
  initializer_range: 0.02


In [2]:
class BertEmbeddings(nn.Module):
    """Construct the embeddings from word, position and token_type embeddings."""
    def __init__(self, config):
        super().__init__()
        self.word_embeddings = nn.Embedding(config.vocab_size, config.hidden_size, padding_idx=0)
        self.position_embeddings = nn.Embedding(config.max_position_embeddings, config.hidden_size)
        self.token_type_embeddings = nn.Embedding(config.type_vocab_size, config.hidden_size)

        # This provides compatibility with older versions of PyTorch where LayerNorm was not available
        # or for custom LayerNorm implementations. For recent PyTorch, nn.LayerNorm is standard.
        self.LayerNorm = nn.LayerNorm(config.hidden_size, eps=1e-12)
        self.dropout = nn.Dropout(config.hidden_dropout_prob)

    def forward(self, input_ids=None, token_type_ids=None, position_ids=None, inputs_embeds=None):
        if input_ids is not None:
            input_shape = input_ids.size()
        else:
            input_shape = inputs_embeds.size()[:-1]

        seq_length = input_shape[1]

        if position_ids is None:
            position_ids = torch.arange(seq_length, dtype=torch.long, device=self.word_embeddings.weight.device)
            position_ids = position_ids.unsqueeze(0).expand(input_shape)

        if token_type_ids is None:
            token_type_ids = torch.zeros(input_shape, dtype=torch.long, device=self.word_embeddings.weight.device)

        if inputs_embeds is None:
            inputs_embeds = self.word_embeddings(input_ids)

        position_embeddings = self.position_embeddings(position_ids)
        token_type_embeddings = self.token_type_embeddings(token_type_ids)

        embeddings = inputs_embeds + position_embeddings + token_type_embeddings
        embeddings = self.LayerNorm(embeddings)
        embeddings = self.dropout(embeddings)
        return embeddings

# Let's test my BertEmbeddings class with some dummy data to ensure it works as expected.
# I'll create some random input_ids and token_type_ids.

dummy_input_ids = torch.randint(0, bert_config.vocab_size, (1, 512)) # Batch size 1, sequence length 512
dummy_token_type_ids = torch.randint(0, bert_config.type_vocab_size, (1, 512))

# Instantiate the embeddings layer
embeddings_layer = BertEmbeddings(bert_config)

# Pass the dummy data through the embeddings layer
dummy_embeddings = embeddings_layer(input_ids=dummy_input_ids, token_type_ids=dummy_token_type_ids)

print(f"Shape of dummy_input_ids: {dummy_input_ids.shape}")
print(f"Shape of dummy_token_type_ids: {dummy_token_type_ids.shape}")
print(f"Shape of output embeddings: {dummy_embeddings.shape}")


Shape of dummy_input_ids: torch.Size([1, 512])
Shape of dummy_token_type_ids: torch.Size([1, 512])
Shape of output embeddings: torch.Size([1, 512, 768])


In [3]:
class BertSelfAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        if config.hidden_size % config.num_attention_heads != 0 and not hasattr(config, "embedding_size"):
            raise ValueError(
                f"The hidden size ({config.hidden_size}) is not a multiple of the number of attention "
                f"heads ({config.num_attention_heads})"
            )

        self.num_attention_heads = config.num_attention_heads
        self.attention_head_size = int(config.hidden_size / config.num_attention_heads)
        self.all_head_size = self.num_attention_heads * self.attention_head_size

        self.query = nn.Linear(config.hidden_size, self.all_head_size)
        self.key = nn.Linear(config.hidden_size, self.all_head_size)
        self.value = nn.Linear(config.hidden_size, self.all_head_size)

        self.dropout = nn.Dropout(config.attention_probs_dropout_prob)

    def transpose_for_scores(self, x):
        new_x_shape = x.size()[:-1] + (self.num_attention_heads, self.attention_head_size)
        x = x.view(new_x_shape)
        return x.permute(0, 2, 1, 3) # (batch_size, num_heads, seq_length, head_size)

    def forward(
        self, hidden_states, attention_mask=None
    ):
        mixed_query_layer = self.query(hidden_states)
        mixed_key_layer = self.key(hidden_states)
        mixed_value_layer = self.value(hidden_states)

        query_layer = self.transpose_for_scores(mixed_query_layer)
        key_layer = self.transpose_for_scores(mixed_key_layer)
        value_layer = self.transpose_for_scores(mixed_value_layer)

        # Take the dot product between "query" and "key" to get the raw attention scores.
        attention_scores = torch.matmul(query_layer, key_layer.transpose(-1, -2))
        attention_scores = attention_scores / math.sqrt(self.attention_head_size)

        if attention_mask is not None:
            # Apply the attention mask is (precomputed for all layers in BertModel forward() function)
            attention_scores = attention_scores + attention_mask

        # Normalize the attention scores to probabilities.
        attention_probs = nn.Softmax(dim=-1)(attention_scores)

        # This is ostensibly for attention dropout, but it is applied to the attention probabilities, not the scores.
        attention_probs = self.dropout(attention_probs)

        context_layer = torch.matmul(attention_probs, value_layer)

        context_layer = context_layer.permute(0, 2, 1, 3).contiguous()
        new_context_layer_shape = context_layer.size()[:-2] + (self.all_head_size,)
        context_layer = context_layer.view(new_context_layer_shape)

        return context_layer


class BertSelfOutput(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.dense = nn.Linear(config.hidden_size, config.hidden_size)
        self.LayerNorm = nn.LayerNorm(config.hidden_size, eps=1e-12)
        self.dropout = nn.Dropout(config.hidden_dropout_prob)

    def forward(self, hidden_states, input_tensor):
        hidden_states = self.dense(hidden_states)
        hidden_states = self.dropout(hidden_states)
        hidden_states = self.LayerNorm(hidden_states + input_tensor)
        return hidden_states


# Let's test BertSelfAttention and BertSelfOutput with some dummy data
print("\nTesting BertSelfAttention and BertSelfOutput...")
# Dummy input with batch_size=1, seq_length=512, hidden_size=768
dummy_hidden_states = torch.randn(1, 512, bert_config.hidden_size)
# Dummy attention mask (e.g., for padding, 0 for masked positions, -1e9 for unmasked)
# Here, I'll create a mask that doesn't mask anything for simplicity (all zeros)
dummy_attention_mask = torch.zeros(1, 1, 1, 512)

# Instantiate the self-attention layer
self_attention_layer = BertSelfAttention(bert_config)

# Pass the dummy data through the self-attention layer
attention_output = self_attention_layer(dummy_hidden_states, dummy_attention_mask)

print(f"Shape of BertSelfAttention output: {attention_output.shape}")

# Instantiate the self-output layer
self_output_layer = BertSelfOutput(bert_config)

# Pass the attention output and original hidden states through the self-output layer
final_output = self_output_layer(attention_output, dummy_hidden_states)

print(f"Shape of BertSelfOutput final output: {final_output.shape}")


Testing BertSelfAttention and BertSelfOutput...
Shape of BertSelfAttention output: torch.Size([1, 512, 768])
Shape of BertSelfOutput final output: torch.Size([1, 512, 768])


In [4]:
def gelu(x):
    """Implementation of the GELU activation function. Ref: https://arxiv.org/abs/1606.08415"""
    return x * 0.5 * (1.0 + torch.erf(x / math.sqrt(2.0)))


class BertIntermediate(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.dense = nn.Linear(config.hidden_size, config.intermediate_size)
        if isinstance(config.hidden_act, str):
            self.intermediate_act_fn = gelu
        else:
            self.intermediate_act_fn = config.hidden_act

    def forward(self, hidden_states):
        hidden_states = self.dense(hidden_states)
        hidden_states = self.intermediate_act_fn(hidden_states)
        return hidden_states


class BertOutput(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.dense = nn.Linear(config.intermediate_size, config.hidden_size)
        self.LayerNorm = nn.LayerNorm(config.hidden_size, eps=1e-12)
        self.dropout = nn.Dropout(config.hidden_dropout_prob)

    def forward(self, hidden_states, input_tensor):
        hidden_states = self.dense(hidden_states)
        hidden_states = self.dropout(hidden_states)
        hidden_states = self.LayerNorm(hidden_states + input_tensor)
        return hidden_states


# Let's test BertIntermediate and BertOutput with some dummy data
print("\nTesting BertIntermediate and BertOutput...")
# Dummy input with batch_size=1, seq_length=512, hidden_size=768
dummy_hidden_states_ffn = torch.randn(1, 512, bert_config.hidden_size)

# Instantiate the intermediate layer
intermediate_layer = BertIntermediate(bert_config)

# Pass the dummy data through the intermediate layer
intermediate_output = intermediate_layer(dummy_hidden_states_ffn)

print(f"Shape of BertIntermediate output: {intermediate_output.shape}")

# Instantiate the output layer
output_layer_ffn = BertOutput(bert_config)

# Pass the intermediate output and original hidden states through the output layer
final_ffn_output = output_layer_ffn(intermediate_output, dummy_hidden_states_ffn)

print(f"Shape of BertOutput final output: {final_ffn_output.shape}")


Testing BertIntermediate and BertOutput...
Shape of BertIntermediate output: torch.Size([1, 512, 3072])
Shape of BertOutput final output: torch.Size([1, 512, 768])


In [5]:
class BertAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.self = BertSelfAttention(config)
        self.output = BertSelfOutput(config)

    def forward(self, hidden_states, attention_mask=None):
        self_output = self.self(hidden_states, attention_mask)
        attention_output = self.output(self_output, hidden_states)
        return attention_output


class BertLayer(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.attention = BertAttention(config)
        self.intermediate = BertIntermediate(config)
        self.output = BertOutput(config)

    def forward(self, hidden_states, attention_mask=None):
        attention_output = self.attention(hidden_states, attention_mask)
        intermediate_output = self.intermediate(attention_output)
        layer_output = self.output(intermediate_output, attention_output)
        return layer_output


# Let's test the complete BertLayer with some dummy data
print("\nTesting BertLayer...")
# Dummy input with batch_size=1, seq_length=512, hidden_size=768
dummy_hidden_states_layer = torch.randn(1, 512, bert_config.hidden_size)
# Dummy attention mask (all zeros for simplicity, meaning no masking)
dummy_attention_mask_layer = torch.zeros(1, 1, 1, 512)

# Instantiate a BertLayer
bert_layer = BertLayer(bert_config)

# Pass the dummy data through the BertLayer
layer_output = bert_layer(dummy_hidden_states_layer, dummy_attention_mask_layer)

print(f"Shape of BertLayer output: {layer_output.shape}")


Testing BertLayer...
Shape of BertLayer output: torch.Size([1, 512, 768])


In [6]:
class BertEncoder(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.layer = nn.ModuleList([BertLayer(config) for _ in range(config.num_hidden_layers)])

    def forward(self, hidden_states, attention_mask=None):
        for i, layer_module in enumerate(self.layer):
            hidden_states = layer_module(hidden_states, attention_mask)
        return hidden_states


# Let's test the complete BertEncoder with some dummy data
print("\nTesting BertEncoder...")
# Dummy input with batch_size=1, seq_length=512, hidden_size=768
dummy_encoder_input = torch.randn(1, 512, bert_config.hidden_size)
# Dummy attention mask (all zeros for simplicity)
dummy_encoder_mask = torch.zeros(1, 1, 1, 512)

# Instantiate the BertEncoder
bert_encoder = BertEncoder(bert_config)

# Pass the dummy data through the BertEncoder
encoder_output = bert_encoder(dummy_encoder_input, dummy_encoder_mask)

print(f"Shape of BertEncoder output: {encoder_output.shape}")


Testing BertEncoder...
Shape of BertEncoder output: torch.Size([1, 512, 768])


In [7]:
class BertPooler(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.dense = nn.Linear(config.hidden_size, config.hidden_size)
        self.activation = nn.Tanh()

    def forward(self, hidden_states):
        # We take the first token (CLS token) hidden state
        first_token_tensor = hidden_states[:, 0]
        pooled_output = self.dense(first_token_tensor)
        pooled_output = self.activation(pooled_output)
        return pooled_output


class BertPredictionHeadTransform(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.dense = nn.Linear(config.hidden_size, config.hidden_size)
        if isinstance(config.hidden_act, str):
            self.transform_act_fn = gelu
        else:
            self.transform_act_fn = config.hidden_act
        self.LayerNorm = nn.LayerNorm(config.hidden_size, eps=1e-12)

    def forward(self, hidden_states):
        hidden_states = self.dense(hidden_states)
        hidden_states = self.transform_act_fn(hidden_states)
        hidden_states = self.LayerNorm(hidden_states)
        return hidden_states


class BertLMPredictionHead(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.transform = BertPredictionHeadTransform(config)

        # The output weights are tied to the input embeddings' weights, so I'll create them later if needed.
        self.decoder = nn.Linear(config.hidden_size, config.vocab_size, bias=False)
        self.bias = nn.Parameter(torch.zeros(config.vocab_size))

        # Need to tie the weights later, once the embeddings are initialized.
        # self.decoder.weight = word_embeddings_weight

    def forward(self, hidden_states, word_embeddings_weight=None):
        hidden_states = self.transform(hidden_states)
        if word_embeddings_weight is not None:
            # If word embeddings weight is provided, use it for the decoder layer
            # This is how weights are tied in BERT for MLM head.
            # The decoder's weight matrix is the transpose of the word embedding matrix.
            # This is a common practice to reduce the number of parameters.
            output = torch.matmul(hidden_states, word_embeddings_weight.transpose(0, 1)) + self.bias
        else:
            output = self.decoder(hidden_states) + self.bias
        return output

class BertOnlyMLMHead(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.predictions = BertLMPredictionHead(config)

    def forward(self, sequence_output, word_embeddings_weight=None):
        prediction_scores = self.predictions(sequence_output, word_embeddings_weight)
        return prediction_scores


# Let's test the MLM head components
print("\nTesting BertPooler and BertOnlyMLMHead...")
# Dummy encoder output from the last BertEncoder test (batch_size, seq_length, hidden_size)
dummy_encoder_output = encoder_output # Using the output from the previous BertEncoder test

# Instantiate BertPooler
bert_pooler = BertPooler(bert_config)
pooled_output = bert_pooler(dummy_encoder_output)
print(f"Shape of BertPooler output: {pooled_output.shape}")

# Instantiate BertOnlyMLMHead
bert_mlm_head = BertOnlyMLMHead(bert_config)

# For testing, I'll pass a dummy word_embeddings_weight
# In a full model, this would come from the BertEmbeddings.word_embeddings.weight
dummy_word_embeddings_weight = torch.randn(bert_config.vocab_size, bert_config.hidden_size)

prediction_scores = bert_mlm_head(dummy_encoder_output, dummy_word_embeddings_weight)
print(f"Shape of BertOnlyMLMHead output (prediction scores): {prediction_scores.shape}")


Testing BertPooler and BertOnlyMLMHead...
Shape of BertPooler output: torch.Size([1, 768])
Shape of BertOnlyMLMHead output (prediction scores): torch.Size([1, 512, 30522])


In [8]:
class BertOnlyNSPHead(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.seq_relationship = nn.Linear(config.hidden_size, 2) # Two classes: IsNextSentence, NotNextSentence

    def forward(self, pooled_output):
        seq_relationship_score = self.seq_relationship(pooled_output)
        return seq_relationship_score


# Let's test the NSP head
print("\nTesting BertOnlyNSPHead...")
# Use the pooled_output from the previous BertPooler test
dummy_pooled_output = pooled_output

# Instantiate BertOnlyNSPHead
bert_nsp_head = BertOnlyNSPHead(bert_config)

# Pass the dummy pooled output through the NSP head
sequence_relationship_scores = bert_nsp_head(dummy_pooled_output)

print(f"Shape of BertOnlyNSPHead output (sequence relationship scores): {sequence_relationship_scores.shape}")


Testing BertOnlyNSPHead...
Shape of BertOnlyNSPHead output (sequence relationship scores): torch.Size([1, 2])


In [9]:
class BertModel(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.embeddings = BertEmbeddings(config)
        self.encoder = BertEncoder(config)
        self.pooler = BertPooler(config)

        # Initialize prediction heads
        self.cls = BertOnlyMLMHead(config)
        self.cls_nsp = BertOnlyNSPHead(config)

        # Tie weights: The output weights of the decoder in the MLM head are tied to the input word embeddings.
        # This is typically done after initializing both the embedding layer and the prediction head.
        self.cls.predictions.decoder.weight = self.embeddings.word_embeddings.weight


    def forward(self, input_ids=None, attention_mask=None, token_type_ids=None, position_ids=None):
        # 1. Generate initial embeddings
        embedding_output = self.embeddings(input_ids=input_ids, position_ids=position_ids, token_type_ids=token_type_ids)

        # If no attention mask is provided, create a default one (all ones, meaning no masking)
        if attention_mask is None:
            attention_mask = torch.ones(input_ids.shape, device=input_ids.device)

        # BERT's attention mask is a 2D mask. I need to convert it to a 4D mask for the attention layers.
        # The mask is reshaped to `(batch_size, 1, 1, seq_length)` to be broadcastable.
        # The elements where the mask is 0 (padding) should become a very large negative number (e.g., -1e9).
        extended_attention_mask = attention_mask.unsqueeze(1).unsqueeze(2)
        extended_attention_mask = (1.0 - extended_attention_mask) * -10000.0

        # 2. Pass embeddings through the encoder layers
        sequence_output = self.encoder(embedding_output, extended_attention_mask)

        # 3. Get pooled output for NSP
        pooled_output = self.pooler(sequence_output)

        # 4. Get MLM prediction scores
        prediction_scores = self.cls(sequence_output, self.embeddings.word_embeddings.weight)

        # 5. Get NSP prediction scores
        seq_relationship_score = self.cls_nsp(pooled_output)

        return prediction_scores, seq_relationship_score


# Let's test the complete BertModel with some dummy data
print("\nTesting BertModel...")

# Dummy input with batch_size=1, seq_length=512
dummy_input_ids = torch.randint(0, bert_config.vocab_size, (1, 512))
dummy_token_type_ids = torch.randint(0, bert_config.type_vocab_size, (1, 512))
dummy_attention_mask = torch.ones(1, 512) # No masking for simplicity in this test

# Instantiate the BertModel
bert_model = BertModel(bert_config)

# Pass the dummy data through the BertModel
mlm_scores, nsp_scores = bert_model(
    input_ids=dummy_input_ids,
    attention_mask=dummy_attention_mask,
    token_type_ids=dummy_token_type_ids
)

print(f"Shape of BertModel MLM prediction scores: {mlm_scores.shape}")
print(f"Shape of BertModel NSP prediction scores: {nsp_scores.shape}")


Testing BertModel...
Shape of BertModel MLM prediction scores: torch.Size([1, 512, 30522])
Shape of BertModel NSP prediction scores: torch.Size([1, 2])


In [10]:
# Let's create a simple corpus for demonstration purposes.
corpus = [
    "The quick brown fox jumps over the lazy dog.",
    "The dog barks loudly at the cat.",
    "A cat sat on the mat.",
    "The fox is a clever animal.",
    "This is an example sentence for BERT pre-training."
]

# Define special tokens and vocabulary mapping.
# In a real scenario, I would use a pre-trained WordPiece tokenizer.
# For this example, I'll simulate a very basic one.
vocabulary = {
    "[PAD]": 0, "[CLS]": 1, "[SEP]": 2, "[MASK]": 3, "[UNK]": 4,
    "the": 5, "quick": 6, "brown": 7, "fox": 8, "jumps": 9,
    "over": 10, "lazy": 11, "dog": 12, "barks": 13, "loudly": 14,
    "at": 15, "cat": 16, "a": 17, "sat": 18, "on": 19, "mat": 20,
    "is": 21, "clever": 22, "animal": 23, "this": 24, "an": 25,
    "example": 26, "sentence": 27, "for": 28, "bert": 29, "pre-training": 30
}

# Reverse mapping for visualization (optional)
id_to_token = {v: k for k, v in vocabulary.items()}

# A simple tokenization function
def simple_tokenizer(text):
    return [token.lower() for token in text.replace('.', '').split()]

# Function to convert tokens to input IDs, handling unknown words
def encode_tokens(tokens, vocab):
    return [vocab.get(token, vocab["[UNK]"]) for token in tokens]

print("Sample corpus:")
for i, sent in enumerate(corpus):
    print(f"  {i+1}. {sent}")

print("\nExample tokenization and encoding:")
sample_text = corpus[0]
sample_tokens = simple_tokenizer(sample_text)
sample_ids = encode_tokens(sample_tokens, vocabulary)
print(f"Text: '{sample_text}'")
print(f"Tokens: {sample_tokens}")
print(f"IDs: {sample_ids}")


# Now, I'll create a function to prepare input for BERT, including special tokens, segment IDs, and attention mask.
# This function will be crucial for generating samples for both MLM and NSP.
def prepare_bert_input(sentence_a, sentence_b=None, max_seq_length=bert_config.max_position_embeddings):
    tokens_a = simple_tokenizer(sentence_a)
    tokens = ["[CLS]"] + tokens_a + ["[SEP]"]
    token_type_ids = [0] * (len(tokens_a) + 2)

    if sentence_b:
        tokens_b = simple_tokenizer(sentence_b)
        tokens += tokens_b + ["[SEP]"]
        token_type_ids += [1] * (len(tokens_b) + 1)

    # Truncate if longer than max_seq_length
    if len(tokens) > max_seq_length:
        tokens = tokens[:max_seq_length]
        token_type_ids = token_type_ids[:max_seq_length]

    input_ids = encode_tokens(tokens, vocabulary)

    # Pad to max_seq_length
    attention_mask = [1] * len(input_ids)
    padding_length = max_seq_length - len(input_ids)
    input_ids = input_ids + [vocabulary["[PAD]"]] * padding_length
    attention_mask = attention_mask + [0] * padding_length
    token_type_ids = token_type_ids + [0] * padding_length

    return {
        'input_ids': torch.tensor([input_ids], dtype=torch.long),
        'token_type_ids': torch.tensor([token_type_ids], dtype=torch.long),
        'attention_mask': torch.tensor([attention_mask], dtype=torch.long),
        'original_tokens': tokens # Keep original tokens for MLM label comparison
    }

print("\nTesting prepare_bert_input for a single sentence:")
single_sentence_input = prepare_bert_input(corpus[0], max_seq_length=15)
print(f"Input IDs: {single_sentence_input['input_ids']}")
print(f"Token Type IDs: {single_sentence_input['token_type_ids']}")
print(f"Attention Mask: {single_sentence_input['attention_mask']}")
print(f"Original Tokens: {single_sentence_input['original_tokens']}")

print("\nTesting prepare_bert_input for a sentence pair:")
sentence_pair_input = prepare_bert_input(corpus[0], corpus[1], max_seq_length=20)
print(f"Input IDs: {sentence_pair_input['input_ids']}")
print(f"Token Type IDs: {sentence_pair_input['token_type_ids']}")
print(f"Attention Mask: {sentence_pair_input['attention_mask']}")
print(f"Original Tokens: {sentence_pair_input['original_tokens']}")


Sample corpus:
  1. The quick brown fox jumps over the lazy dog.
  2. The dog barks loudly at the cat.
  3. A cat sat on the mat.
  4. The fox is a clever animal.
  5. This is an example sentence for BERT pre-training.

Example tokenization and encoding:
Text: 'The quick brown fox jumps over the lazy dog.'
Tokens: ['the', 'quick', 'brown', 'fox', 'jumps', 'over', 'the', 'lazy', 'dog']
IDs: [5, 6, 7, 8, 9, 10, 5, 11, 12]

Testing prepare_bert_input for a single sentence:
Input IDs: tensor([[ 1,  5,  6,  7,  8,  9, 10,  5, 11, 12,  2,  0,  0,  0,  0]])
Token Type IDs: tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]])
Attention Mask: tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0]])
Original Tokens: ['[CLS]', 'the', 'quick', 'brown', 'fox', 'jumps', 'over', 'the', 'lazy', 'dog', '[SEP]']

Testing prepare_bert_input for a sentence pair:
Input IDs: tensor([[ 1,  5,  6,  7,  8,  9, 10,  5, 11, 12,  2,  5, 12, 13, 14, 15,  5, 16,
          2,  0]])
Token Type IDs: tensor([[0, 0, 0

In [11]:
import random

def mask_tokens(input_ids, vocab, mask_prob=0.15):
    labels = input_ids.clone() # Create a copy for labels; original token IDs at masked positions
    masked_input_ids = input_ids.clone()

    # Special tokens to exclude from masking
    special_tokens_ids = [vocab["[PAD]"], vocab["[CLS]"], vocab["[SEP]"]]

    # Get positions of tokens that can be masked
    candidate_mask_indices = []
    for i, token_id in enumerate(input_ids[0].tolist()): # Assuming batch size 1 for simplicity here
        if token_id not in special_tokens_ids:
            candidate_mask_indices.append(i)

    random.shuffle(candidate_mask_indices)

    num_mask = min(len(candidate_mask_indices), int(len(input_ids[0]) * mask_prob))
    masked_indices = candidate_mask_indices[:num_mask]

    for index in masked_indices:
        # Labels are 100 for non-masked tokens, actual token ID for masked tokens
        # PyTorch's CrossEntropyLoss ignores targets with value -100
        labels[0, index] = input_ids[0, index]

        # 80% of the time, replace with [MASK]
        if random.random() < 0.8:
            masked_input_ids[0, index] = vocab["[MASK]"]
        # 10% of the time, replace with a random word
        elif random.random() < 0.5: # 0.8 + 0.1 = 0.9, so 0.1 probability here
            random_word_id = random.randint(0, len(vocab) - 1)
            masked_input_ids[0, index] = random_word_id
        # 10% of the time, keep original word (do nothing)
        # else: masked_input_ids[0, index] remains input_ids[0, index]

    # For tokens not masked, set their label to -100 so they are ignored in loss calculation
    labels[masked_input_ids == input_ids] = -100
    # Special tokens should also not contribute to the loss
    for token_id in special_tokens_ids:
        labels[input_ids == token_id] = -100

    return masked_input_ids, labels


# Let's test the mask_tokens function with a sample input from our corpus
print("\nTesting mask_tokens for MLM data generation...")

sample_sentence = corpus[0]
single_sentence_prepared = prepare_bert_input(sample_sentence, max_seq_length=15)

original_input_ids = single_sentence_prepared['input_ids']
original_tokens = single_sentence_prepared['original_tokens']

print(f"Original Input IDs: {original_input_ids}")
print(f"Original Tokens: {original_tokens}")

masked_input_ids, mlm_labels = mask_tokens(original_input_ids, vocabulary, mask_prob=0.4) # Increased mask_prob for better visibility in example

print(f"Masked Input IDs: {masked_input_ids}")
print(f"MLM Labels: {mlm_labels}")

# Let's decode the masked input and labels for better understanding
decoded_masked_input = [id_to_token.get(idx.item(), '[UNK]') for idx in masked_input_ids[0]]
decoded_mlm_labels = [id_to_token.get(idx.item(), '[IGN]') if idx != -100 else '[IGN]' for idx in mlm_labels[0]]

print(f"Decoded Masked Input: {' '.join(decoded_masked_input)}")
print(f"Decoded MLM Labels: {' '.join(decoded_mlm_labels)}")



Testing mask_tokens for MLM data generation...
Original Input IDs: tensor([[ 1,  5,  6,  7,  8,  9, 10,  5, 11, 12,  2,  0,  0,  0,  0]])
Original Tokens: ['[CLS]', 'the', 'quick', 'brown', 'fox', 'jumps', 'over', 'the', 'lazy', 'dog', '[SEP]']
Masked Input IDs: tensor([[ 1, 17,  6,  7,  3,  9,  3,  3,  3,  3,  2,  0,  0,  0,  0]])
MLM Labels: tensor([[-100,    5, -100, -100,    8, -100,   10,    5,   11,   12, -100, -100,
         -100, -100, -100]])
Decoded Masked Input: [CLS] a quick brown [MASK] jumps [MASK] [MASK] [MASK] [MASK] [SEP] [PAD] [PAD] [PAD] [PAD]
Decoded MLM Labels: [IGN] the [IGN] [IGN] fox [IGN] over the lazy dog [IGN] [IGN] [IGN] [IGN] [IGN]


In [12]:
import random

def create_pretraining_sample(corpus, vocab, bert_config):
    # 50% chance for IS_NEXT_SENTENCE, 50% for NOT_NEXT_SENTENCE
    is_next_sentence_label = random.choice([0, 1]) # 0 for is_next, 1 for not_next

    # Select a random sentence A
    idx_a = random.randint(0, len(corpus) - 1)
    sentence_a = corpus[idx_a]

    if is_next_sentence_label == 0: # IsNextSentence
        # Try to find the actual next sentence
        if idx_a + 1 < len(corpus):
            sentence_b = corpus[idx_a + 1]
        else:
            # If A is the last sentence, just pick a random sentence for B and flip the label
            # This case makes the dataset generation simpler for a small corpus
            sentence_b = random.choice(corpus)
            is_next_sentence_label = 1
    else: # NotNextSentence
        # Pick a random sentence B that is not immediately after A
        while True:
            idx_b = random.randint(0, len(corpus) - 1)
            if idx_b != idx_a and idx_b != idx_a + 1:
                sentence_b = corpus[idx_b]
                break

    # Prepare BERT input (tokenization, special tokens, segment IDs, attention mask)
    prepared_input = prepare_bert_input(
        sentence_a,
        sentence_b if sentence_b else None, # Pass sentence_b only if it exists
        max_seq_length=bert_config.max_position_embeddings
    )

    # Mask tokens for MLM
    masked_input_ids, mlm_labels = mask_tokens(prepared_input['input_ids'], vocab)

    return {
        'input_ids': masked_input_ids.squeeze(0),
        'token_type_ids': prepared_input['token_type_ids'].squeeze(0),
        'attention_mask': prepared_input['attention_mask'].squeeze(0),
        'mlm_labels': mlm_labels.squeeze(0),
        'nsp_label': torch.tensor(is_next_sentence_label, dtype=torch.long)
    }

# Let's test the data generation function
print("\nTesting create_pretraining_sample function...")
sample_data = create_pretraining_sample(corpus, vocabulary, bert_config)

print(f"Input IDs shape: {sample_data['input_ids'].shape}")
print(f"Token Type IDs shape: {sample_data['token_type_ids'].shape}")
print(f"Attention Mask shape: {sample_data['attention_mask'].shape}")
print(f"MLM Labels shape: {sample_data['mlm_labels'].shape}")
print(f"NSP Label: {sample_data['nsp_label'].item()}")

# Decode a sample for verification
decoded_input = [id_to_token.get(idx.item(), '[UNK]') for idx in sample_data['input_ids']]
decoded_mlm_labels = [id_to_token.get(idx.item(), '[IGN]') if idx != -100 else '[IGN]' for idx in sample_data['mlm_labels']]

print(f"\nDecoded Masked Input (first 20): {' '.join(decoded_input[:20])}")
print(f"Decoded MLM Labels (first 20): {' '.join(decoded_mlm_labels[:20])}")
print(f"NSP Label: {'IsNextSentence' if sample_data['nsp_label'].item() == 0 else 'NotNextSentence'}")



Testing create_pretraining_sample function...
Input IDs shape: torch.Size([512])
Token Type IDs shape: torch.Size([512])
Attention Mask shape: torch.Size([512])
MLM Labels shape: torch.Size([512])
NSP Label: 1

Decoded Masked Input (first 20): [CLS] [MASK] [MASK] [MASK] [MASK] [MASK] [MASK] [SEP] [MASK] [MASK] [MASK] animal [MASK] [MASK] [MASK] [SEP] [PAD] [PAD] [PAD] [PAD]
Decoded MLM Labels (first 20): [IGN] a cat sat on the mat [IGN] the dog barks loudly at the cat [IGN] [IGN] [IGN] [IGN] [IGN]
NSP Label: NotNextSentence


In [19]:
from torch.utils.data import Dataset, DataLoader

class BertPretrainingDataset(Dataset):
    def __init__(self, corpus, vocab, bert_config, num_samples=1000):
        self.corpus = corpus
        self.vocab = vocab
        self.bert_config = bert_config
        self.num_samples = num_samples # Number of samples to generate for pre-training

    def __len__(self):
        return self.num_samples

    def __getitem__(self, idx):
        # For demonstration, I'll generate a fresh sample each time.
        # In a real scenario with a very large corpus, you might pre-process and save samples,
        # or implement more sophisticated on-the-fly generation if memory is a concern.
        sample = create_pretraining_sample(
            self.corpus,
            self.vocab,
            self.bert_config
        )
        return (
            sample['input_ids'],
            sample['token_type_ids'],
            sample['attention_mask'],
            sample['mlm_labels'],
            sample['nsp_label']
        )

# Let's create an instance of the dataset
print("\nCreating BertPretrainingDataset...")
# Increasing num_samples to simulate a larger dataset for more training steps
pretraining_dataset = BertPretrainingDataset(corpus, vocabulary, bert_config, num_samples=500)

# Let's verify by loading a single sample
print(f"Dataset size: {len(pretraining_dataset)}")
sample_input_ids, sample_token_type_ids, sample_attention_mask, sample_mlm_labels, sample_nsp_label = pretraining_dataset[0]

print("\nVerifying a single sample from the dataset:")
print(f"Input IDs shape: {sample_input_ids.shape}")
print(f"Token Type IDs shape: {sample_token_type_ids.shape}")
print(f"Attention Mask shape: {sample_attention_mask.shape}")
print(f"MLM Labels shape: {sample_mlm_labels.shape}")
print(f"NSP Label: {sample_nsp_label.item()}")



Creating BertPretrainingDataset...
Dataset size: 500

Verifying a single sample from the dataset:
Input IDs shape: torch.Size([512])
Token Type IDs shape: torch.Size([512])
Attention Mask shape: torch.Size([512])
MLM Labels shape: torch.Size([512])
NSP Label: 0


In [ ]:
from torch.utils.data import DataLoader
from torch.optim import AdamW

# --- Training Configuration ---
epochs = 5  # Increased epochs for a longer simulated training
batch_size = 10 # Small batch size for our tiny dataset
learning_rate = 1e-4

# --- Device Setup ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# --- Data Loader ---
# Using the BertPretrainingDataset created previously
pretraining_dataloader = DataLoader(
    pretraining_dataset,
    batch_size=batch_size,
    shuffle=True # Shuffle data for better training
)

# --- Model and Optimizer/Loss Setup ---
model = bert_model.to(device) # Move model to selected device
optimizer = AdamW(model.parameters(), lr=learning_rate)

# Loss functions for MLM and NSP
# CrossEntropyLoss for MLM (predicting vocab token) and NSP (binary classification)
mlm_loss_fn = nn.CrossEntropyLoss(ignore_index=-100) # -100 is the ignore index for padded/unmasked tokens
nsp_loss_fn = nn.CrossEntropyLoss() # For binary classification (is_next, not_next)

# --- Training Loop ---
print("\nStarting training loop...")

model.train() # Set the model to training mode
for epoch in range(epochs):
    total_mlm_loss = 0
    total_nsp_loss = 0
    total_loss = 0
    for step, batch in enumerate(pretraining_dataloader):
        # Move batch to device
        input_ids, token_type_ids, attention_mask, mlm_labels, nsp_label = [
            t.to(device) for t in batch
        ]

        optimizer.zero_grad() # Clear previous gradients

        # Forward pass
        mlm_scores, nsp_scores = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids
        )

        # Calculate MLM loss
        # Reshape mlm_scores to (batch_size * sequence_length, vocab_size)
        # Reshape mlm_labels to (batch_size * sequence_length)
        mlm_loss = mlm_loss_fn(mlm_scores.view(-1, bert_config.vocab_size), mlm_labels.view(-1))

        # Calculate NSP loss
        nsp_loss = nsp_loss_fn(nsp_scores.view(-1, 2), nsp_label.view(-1))

        # Combine losses (you might weight these differently in a real scenario)
        loss = mlm_loss + nsp_loss

        # Backward pass and optimization
        loss.backward()
        optimizer.step()

        total_mlm_loss += mlm_loss.item()
        total_nsp_loss += nsp_loss.item()
        total_loss += loss.item()

        # Print progress for every step to ensure visibility with small datasets
        print(f"  Epoch {epoch+1}, Step {step}: Total Loss: {loss.item():.4f}, MLM Loss: {mlm_loss.item():.4f}, NSP Loss: {nsp_loss.item():.4f}")

    avg_mlm_loss = total_mlm_loss / len(pretraining_dataloader)
    avg_nsp_loss = total_nsp_loss / len(pretraining_dataloader)
    avg_total_loss = total_loss / len(pretraining_dataloader)
    print(f"Epoch {epoch+1} finished. Average Total Loss: {avg_total_loss:.4f}, Avg MLM Loss: {avg_mlm_loss:.4f}, Avg NSP Loss: {avg_nsp_loss:.4f}")

print("Training complete!")

In [18]:
print("\n--- Performing Inference with the Trained BERT Model ---")

# Set the model to evaluation mode
model.eval()

# --- Inference for Masked Language Model (MLM) ---
print("\nMLM Inference Example:")
original_sentence_for_mlm = "The quick brown fox jumps over the lazy dog."
# Tokenize the original sentence
base_tokens_for_mlm = simple_tokenizer(original_sentence_for_mlm)

# Decide which token to mask (e.g., 'fox' is at index 3 in base_tokens_for_mlm)
token_to_mask_idx_in_base = 3
# The actual index of the masked token in the BERT input sequence (including [CLS])
mask_idx = 1 + token_to_mask_idx_in_base

# Create the full token list with special tokens and the [MASK] token for input
displayed_mlm_input_tokens = (
    ["[CLS]"] +
    base_tokens_for_mlm[:token_to_mask_idx_in_base] +
    ["[MASK]"] +
    base_tokens_for_mlm[token_to_mask_idx_in_base+1:] +
    ["[SEP]"]
)

# Encode the tokens to input_ids
mlm_input_ids_list = encode_tokens(displayed_mlm_input_tokens, vocabulary)

# Pad to max_seq_length
current_len = len(mlm_input_ids_list)
padding_length = bert_config.max_position_embeddings - current_len
mlm_input_ids_list.extend([vocabulary["[PAD]"]] * padding_length)
mlm_input_ids = torch.tensor([mlm_input_ids_list], dtype=torch.long)

# Create token_type_ids and attention_mask
mlm_token_type_ids_list = [0] * current_len
mlm_token_type_ids_list.extend([0] * padding_length)
mlm_token_type_ids = torch.tensor([mlm_token_type_ids_list], dtype=torch.long)

mlm_attention_mask_list = [1] * current_len
mlm_attention_mask_list.extend([0] * padding_length)
mlm_attention_mask = torch.tensor([mlm_attention_mask_list], dtype=torch.long)

# Move to device
mlm_input_ids = mlm_input_ids.to(device)
mlm_token_type_ids = mlm_token_type_ids.to(device)
mlm_attention_mask = mlm_attention_mask.to(device)

with torch.no_grad(): # Disable gradient calculations for inference
    mlm_scores, _ = model(input_ids=mlm_input_ids, attention_mask=mlm_attention_mask, token_type_ids=mlm_token_type_ids)

# Get predictions for the masked token
predicted_token_scores = mlm_scores[0, mask_idx]
top_k_scores, top_k_indices = torch.topk(predicted_token_scores, k=5)

print(f"Original sentence: '{original_sentence_for_mlm}'")
print(f"Input sequence for MLM: {' '.join(displayed_mlm_input_tokens)}")
print(f"Predicted tokens for [MASK] at index {mask_idx} (original word was '{base_tokens_for_mlm[token_to_mask_idx_in_base]}'):")
for i, (score, index) in enumerate(zip(top_k_scores, top_k_indices)):
    # Use .get() with a default '[UNK]' to handle predicted IDs not in our small vocabulary
    print(f"  {i+1}. {id_to_token.get(index.item(), '[UNK]')} (Score: {score.item():.2f})")


# --- Inference for Next Sentence Prediction (NSP) ---
print("\nNSP Inference Examples:")

# Example 1: Is Next Sentence (True)
nsp_sentence_a_true = "The quick brown fox jumps over the lazy dog."
nsp_sentence_b_true = "The dog barks loudly at the cat."

nsp_input_true = prepare_bert_input(nsp_sentence_a_true, nsp_sentence_b_true)
nsp_input_ids_true = nsp_input_true['input_ids'].to(device)
nsp_token_type_ids_true = nsp_input_true['token_type_ids'].to(device)
nsp_attention_mask_true = nsp_input_true['attention_mask'].to(device)

with torch.no_grad():
    _, nsp_scores_true = model(input_ids=nsp_input_ids_true, attention_mask=nsp_attention_mask_true, token_type_ids=nsp_token_type_ids_true)

nsp_prediction_true = torch.argmax(nsp_scores_true, dim=-1).item()
print(f"Sentence A: '{nsp_sentence_a_true}'")
print(f"Sentence B: '{nsp_sentence_b_true}'")
print(f"NSP Prediction: {'IsNextSentence' if nsp_prediction_true == 0 else 'NotNextSentence'} (Score: {nsp_scores_true[0, nsp_prediction_true].item():.2f})")

# Example 2: Not Next Sentence (False)
nsp_sentence_a_false = "The quick brown fox jumps over the lazy dog."
nsp_sentence_b_false = "A cat sat on the mat."

nsp_input_false = prepare_bert_input(nsp_sentence_a_false, nsp_sentence_b_false)
nsp_input_ids_false = nsp_input_false['input_ids'].to(device)
nsp_token_type_ids_false = nsp_input_false['token_type_ids'].to(device)
nsp_attention_mask_false = nsp_input_false['attention_mask'].to(device)

with torch.no_grad():
    _, nsp_scores_false = model(input_ids=nsp_input_ids_false, attention_mask=nsp_attention_mask_false, token_type_ids=nsp_token_type_ids_false)

nsp_prediction_false = torch.argmax(nsp_scores_false, dim=-1).item()
print(f"\nSentence A: '{nsp_sentence_a_false}'")
print(f"Sentence B: '{nsp_sentence_b_false}'")
print(f"NSP Prediction: {'IsNextSentence' if nsp_prediction_false == 0 else 'NotNextSentence'} (Score: {nsp_scores_false[0, nsp_prediction_false].item():.2f})")


--- Performing Inference with the Trained BERT Model ---

MLM Inference Example:
Original sentence: 'The quick brown fox jumps over the lazy dog.'
Input sequence for MLM: [CLS] the quick brown [MASK] jumps over the lazy dog [SEP]
Predicted tokens for [MASK] at index 4 (original word was 'fox'):
  1. the (Score: 100.77)
  2. [UNK] (Score: 96.42)
  3. [UNK] (Score: 95.39)
  4. [UNK] (Score: 92.71)
  5. [UNK] (Score: 92.71)

NSP Inference Examples:
Sentence A: 'The quick brown fox jumps over the lazy dog.'
Sentence B: 'The dog barks loudly at the cat.'
NSP Prediction: NotNextSentence (Score: 0.02)

Sentence A: 'The quick brown fox jumps over the lazy dog.'
Sentence B: 'A cat sat on the mat.'
NSP Prediction: NotNextSentence (Score: 0.02)
